In [1]:
# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive

drive.mount('/content/drive')

print("✅ Drive mounted.")

Mounted at /content/drive
✅ Drive mounted.


In [3]:
# =========================================================
# INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install -q open_clip_torch
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.7 MB/s eta 0:00:00


In [4]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch

import open_clip

print("✅ Libraries imported.")

✅ Libraries imported.


In [5]:
# =========================================================
# CONFIG
# =========================================================

PROJECT_ROOT = "/content/drive/MyDrive/PersonalFashionStylistV2"

CONFIG = {

    "inventory_csv": os.path.join(
        PROJECT_ROOT,
        "datasets",
        "metadata",
        "fashion_inventory.csv"
    ),

    "embeddings_output": os.path.join(
        PROJECT_ROOT,
        "embeddings",
        "clip_embeddings.npy"
    ),

    "metadata_output": os.path.join(
        PROJECT_ROOT,
        "embeddings",
        "embedding_metadata.csv"
    ),

    "batch_size": 64
}

print("✅ Config initialized.")

✅ Config initialized.


In [6]:
# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("✅ Device:", device)

✅ Device: cuda


In [7]:
# =========================================================
# LOAD INVENTORY DATAFRAME
# =========================================================

inventory_df = pd.read_csv(
    CONFIG["inventory_csv"]
)

print("✅ Inventory loaded.")

print("\nTotal Items:",
      len(inventory_df))

inventory_df.head()

✅ Inventory loaded.

Total Items: 44074


,id,gender,masterCategory,subCategory,articleType,baseColour,season,usage,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,Casual,/kaggle/input/fashion-product-images-small/ima...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,Casual,/kaggle/input/fashion-product-images-small/ima...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,Casual,/kaggle/input/fashion-product-images-small/ima...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,Casual,/kaggle/input/fashion-product-images-small/ima...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,Casual,/kaggle/input/fashion-product-images-small/ima...


In [8]:
# =========================================================
# LOAD OPENCLIP MODEL
# =========================================================

model, _, preprocess = open_clip.create_model_and_transforms(

    "ViT-B-32",

    pretrained="laion2b_s34b_b79k"
)

tokenizer = open_clip.get_tokenizer(
    "ViT-B-32"
)

model = model.to(device)

model.eval()

print("✅ OpenCLIP model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

✅ OpenCLIP model loaded.


In [9]:
# =========================================================
# CHECK FASHION PRODUCT IMAGE DIRECTORY
# =========================================================

fashion_drive_path = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "fashion_products"
)

print(os.listdir(fashion_drive_path))

['images']


In [ ]:
# =========================================================
# DEFINE FASHION PRODUCT DATASET PATH
# =========================================================

fashion_products_path = "/root/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1"

print("✅ Fashion product dataset path set.")

✅ Fashion product dataset path set.


In [10]:
# =========================================================
# REDOWNLOAD FASHION PRODUCT DATASET
# =========================================================

import kagglehub

fashion_products_path = kagglehub.dataset_download(
    "paramaggarwal/fashion-product-images-small"
)

print("✅ Dataset downloaded.")
print(fashion_products_path)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
✅ Dataset downloaded.
/kaggle/input/fashion-product-images-small


In [11]:
# =========================================================
# VERIFY DATASET STRUCTURE
# =========================================================

print(os.listdir(fashion_products_path))

['myntradataset', 'images', 'styles.csv']


In [ ]:
# =========================================================
# COPY FASHION PRODUCT IMAGES TO DRIVE
# =========================================================

import shutil

source_images_dir = os.path.join(
    fashion_products_path,
    "images"
)

destination_images_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "fashion_products",
    "images"
)

if not os.path.exists(destination_images_dir):

    print("🚀 Copying fashion inventory images to Drive...")

    shutil.copytree(
        source_images_dir,
        destination_images_dir
    )

print("✅ Fashion inventory copied to Drive.")

🚀 Copying fashion inventory images to Drive...


KeyboardInterrupt: 

In [ ]:
# =========================================================
# COPY INVENTORY IMAGES TO LOCAL SSD
# =========================================================

import shutil

# Source image directory in Drive
drive_inventory_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "fashion_products",
    "images"
)

# Local SSD directory
local_inventory_dir = "/content/fashion_inventory_images"

# Copy only once
if not os.path.exists(local_inventory_dir):

    print("🚀 Copying inventory images to local SSD...")

    shutil.copytree(
        drive_inventory_dir,
        local_inventory_dir
    )

print("✅ Local inventory images ready.")

In [12]:
# =========================================================
# FIX INVENTORY IMAGE PATHS
# =========================================================

real_image_dir = os.path.join(
    fashion_products_path,
    "images"
)

def fix_inventory_path(old_path):

    filename = os.path.basename(old_path)

    return os.path.join(
        real_image_dir,
        filename
    )

inventory_df["image_path"] = inventory_df["image_path"].apply(
    fix_inventory_path
)

print("✅ Inventory paths fixed.")

inventory_df.head()

✅ Inventory paths fixed.


,id,gender,masterCategory,subCategory,articleType,baseColour,season,usage,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,Casual,/kaggle/input/fashion-product-images-small/ima...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,Casual,/kaggle/input/fashion-product-images-small/ima...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,Casual,/kaggle/input/fashion-product-images-small/ima...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,Casual,/kaggle/input/fashion-product-images-small/ima...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,Casual,/kaggle/input/fashion-product-images-small/ima...


In [13]:
sample_path = inventory_df.iloc[0]["image_path"]

print(sample_path)

print(os.path.exists(sample_path))

/kaggle/input/fashion-product-images-small/images/15970.jpg
True


In [14]:
# =========================================================
# CLIP EMBEDDING DATASET
# =========================================================

from torch.utils.data import Dataset, DataLoader

class ClipEmbeddingDataset(Dataset):

    def __init__(self, dataframe, preprocess):

        self.dataframe = dataframe.reset_index(drop=True)

        self.preprocess = preprocess

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")

        image = self.preprocess(image)

        return image, idx

In [15]:
# =========================================================
# CREATE EMBEDDING DATALOADER
# =========================================================

embedding_dataset = ClipEmbeddingDataset(

    inventory_df,

    preprocess
)

embedding_loader = DataLoader(

    embedding_dataset,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=2,

    pin_memory=True
)

print("✅ Embedding DataLoader ready.")

✅ Embedding DataLoader ready.


In [16]:
# =========================================================
# GENERATE CLIP EMBEDDINGS
# =========================================================

all_embeddings = []

with torch.no_grad():

    for images, indices in tqdm(embedding_loader):

        images = images.to(device)

        # Generate embeddings
        features = model.encode_image(images)

        # Normalize embeddings
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        # Move to CPU numpy
        features = features.cpu().numpy()

        all_embeddings.append(features)

print("✅ Embedding generation complete.")

100%|██████████| 689/689 [04:15<00:00,  2.70it/s]

✅ Embedding generation complete.


In [17]:
# =========================================================
# COMBINE ALL EMBEDDINGS
# =========================================================

all_embeddings = np.vstack(all_embeddings)

print("✅ Final embedding matrix shape:")

print(all_embeddings.shape)

✅ Final embedding matrix shape:
(44074, 512)


In [20]:
# =========================================================
# CREATE OUTPUT DIRECTORIES
# =========================================================

os.makedirs(

    os.path.join(PROJECT_ROOT, "embeddings"),

    exist_ok=True
)

os.makedirs(

    os.path.join(PROJECT_ROOT, "faiss"),

    exist_ok=True
)

print("✅ Output directories ready.")

✅ Output directories ready.


In [21]:
# =========================================================
# SAVE EMBEDDINGS
# =========================================================

np.save(

    CONFIG["embeddings_output"],

    all_embeddings
)

print("✅ Embeddings saved.")

print(CONFIG["embeddings_output"])

✅ Embeddings saved.
/content/drive/MyDrive/PersonalFashionStylistV2/embeddings/clip_embeddings.npy


In [18]:
# =========================================================
# SAVE EMBEDDING METADATA
# =========================================================

inventory_df.to_csv(

    CONFIG["metadata_output"],

    index=False
)

print("✅ Embedding metadata saved.")

print(CONFIG["metadata_output"])

✅ Embedding metadata saved.
/content/drive/MyDrive/PersonalFashionStylistV2/embeddings/embedding_metadata.csv


In [19]:
# =========================================================
# CHECK EMBEDDINGS DIRECTORY
# =========================================================

embeddings_dir = os.path.join(
    PROJECT_ROOT,
    "embeddings"
)

print("Directory Exists:",
      os.path.exists(embeddings_dir))

if os.path.exists(embeddings_dir):

    print("\nFiles:")

    print(os.listdir(embeddings_dir))

Directory Exists: True

Files:
['embedding_metadata.csv']
